# MSigDB — API & bulk

Broad Institute's [Molecular Signatures Database](https://www.gsea-msigdb.org/gsea/msigdb/)
publishes Hallmark plus C1–C8 gene-set collections in GMT, and serves
per-set JSON records via the website's REST endpoint.

| Mode | Functions |
|---|---|
| **Per-set JSON** | `query_gene_set`, `query_genes` |
| **Bulk GMT** | `download_gmt`, `load_gmt` |

In [1]:
from biodb import msigdb

## 1. Per-set JSON — one Hallmark set at a time

`query_gene_set` sniffs content-type rather than trusting the status
code (MSigDB serves an HTML 200 for unknown set names).

In [2]:
record = msigdb.query_gene_set("HALLMARK_APOPTOSIS")
{k: record.get(k) for k in ("systematicName", "msigdbURL", "pmid", "exactSource") if k in record}

{'systematicName': 'M5902',
 'msigdbURL': 'https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_APOPTOSIS',
 'pmid': '26771021',
 'exactSource': ''}

In [3]:
print(f"{len(record['geneSymbols'])} genes; first 8: {record['geneSymbols'][:8]}")

161 genes; first 8: ['ADD1', 'AIFM3', 'ANKH', 'ANXA1', 'APP', 'ATF3', 'AVPR1A', 'BAX']


Convenience wrapper that returns just the gene-symbol list:

In [4]:
msigdb.query_genes("HALLMARK_HYPOXIA")[:10]

['ACKR3',
 'ADM',
 'ADORA2B',
 'AK4',
 'AKAP12',
 'ALDOA',
 'ALDOB',
 'ALDOC',
 'AMPD3',
 'ANGPTL4']

## 2. Bulk GMT — pull a whole collection

Hallmark (`h.all`) is tiny (~50 KB), so the cell below actually
downloads it. Other collections (`c2.all`, `c5.all`, `msigdb` for the
combined dump) are bigger; same call shape.

In [5]:
path = msigdb.download_gmt(collection="h.all")
print(path)

/Users/bschilder/.cache/biodb/msigdb/h.all.v2025.1.Hs.symbols.gmt


`load_gmt` parses to either a long DataFrame or a `{(id, label):
[genes]}` dict.

In [6]:
df = msigdb.load_gmt(collection="h.all", return_format="pandas")
print(df.shape)
df.head()

(7322, 3)


,id,label,gene
0,HALLMARK_ADIPOGENESIS,https://www.gsea-msigdb.org/gsea/msigdb/human/...,ABCA1
1,HALLMARK_ADIPOGENESIS,https://www.gsea-msigdb.org/gsea/msigdb/human/...,ABCB8
2,HALLMARK_ADIPOGENESIS,https://www.gsea-msigdb.org/gsea/msigdb/human/...,ACAA2
3,HALLMARK_ADIPOGENESIS,https://www.gsea-msigdb.org/gsea/msigdb/human/...,ACADL
4,HALLMARK_ADIPOGENESIS,https://www.gsea-msigdb.org/gsea/msigdb/human/...,ACADM


In [7]:
d = msigdb.load_gmt(collection="h.all", return_format="dict")
list(d)[:3], len(next(iter(d.values())))

([('HALLMARK_ADIPOGENESIS',
   'https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ADIPOGENESIS'),
  ('HALLMARK_ALLOGRAFT_REJECTION',
   'https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ALLOGRAFT_REJECTION'),
  ('HALLMARK_ANDROGEN_RESPONSE',
   'https://www.gsea-msigdb.org/gsea/msigdb/human/geneset/HALLMARK_ANDROGEN_RESPONSE')],
 200)